# Demand Calculations
In this notebook, we calculate state level demand for each byproduct across its key marketable end-uses.

In [43]:
import pandas as pd
import numpy as np
import geopandas as gpd
import math

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.transforms import ScaledTranslation
import seaborn as sns
from cmcrameri import cm

from molmass import Formula

import requests

# add main repo directory to path
import sys
import os
sys.path.append(os.path.abspath('..'))

# add submodules to path
sys.path.append('../submodules/gw-avail-and-comp')

# set up figure style in same way as gw-avail-and-comp repo
from project_utils import setup_figure_style, subscript_numbers, get_boundaries, project_crs
setup_figure_style()

# Load Data

In [44]:
# state geodataframe
st_df = get_boundaries('state')
st_df.drop(index=st_df[st_df['STUSPS']=='DC'].index, inplace=True)
st_df = st_df.sort_values(by='NAME').reset_index(drop=True)

In [45]:
# load market, demand, and price data
demand_path = '../data/inputs/sw1_demand.xlsx'

# market - collapse by end use category
market_df = pd.read_excel(demand_path, sheet_name = 'X1. Markets', na_values=['–'])
market_df = market_df.groupby(['byproduct', 'end_use_category'], as_index=False).agg({'pct_demand': lambda x: x.sum(skipna=False)})

# demand
bpd_df = pd.read_excel(demand_path, sheet_name = 'X2. BP x Domestic', na_values=['–'])
bpms_df = pd.read_excel(demand_path, sheet_name = 'X3. BP-Market x State', na_values=['–'])

# price
price_df = pd.read_excel(demand_path, sheet_name = 'X4. Pricing', na_values=['–'])

In [46]:
# disaggregation proxies
proxy_path = '../data/inputs/sw2_disaggregation_proxies.xlsx'
proxy_file = pd.ExcelFile(proxy_path)
sheets_to_read = [sheet for sheet in proxy_file.sheet_names if sheet != "README"]
proxy_dict = {sheet : proxy_file.parse(sheet, header=3, usecols='B:E', nrows=48) for sheet in sheets_to_read}

# identify proxies to be used for each end use category
market_to_proxy_dict = {
    # Manufacturing
    'Cement Manufacturing' : 'cement_production',
    'Lime Manufacturing' : 'lime_facilities',
    'Glass Manufacturing' : 'glass_emissions',
    'Gypsum-Based Product Manufacturing' : 'gypsum_facilities',
    'Steel Pickling' : 'steel_pickling_facilities',
    'Pulp and Paper Manufacturing' : 'pulp_and_paper_facilities',
    'Lithium Battery Manufacturing' : 'lithium_battery_capacity',
    'Cosmetics and Personal Care Product Manufacturing' : 'cosmetics_employment',
    'Organic Chemical Manufacturing' : 'org_chemical_employment',
    'Pharmaceutical Manufacturing' : 'pharma_employment',
    'Phosphoric Acid Manufacturing' : 'phosphoric_acid_facilities',
    'Miscellaneous Chemical Manufacturing' : 'chemical_employment',
    'Miscellaneous Manufacturing' : 'manufacturing_employment',

    # Utilities and Infrastructure
    'Industrial Water Treatment' : 'industrial_water_use',
    'Drinking Water Treatment' : 'public_supply_water_use',
    'Wastewater Treatment' : 'wwtp_flow',
    'Deicing and Dust Control' : 'road_salts_application',
    'Miscellaneous Construction' : 'construction_spending',

    # Energy and Electricity
    'Oil and Gas Drilling' : 'oil_and_gas_wells',
    'Oil and Gas Well Acidification' : 'oil_and_gas_wells',
    'Flue Gas Desulfurization' : 'flue_gas_desulfurization',

    # Agriculture
    'Potash Fertilizer Application' : 'potash_fertilizer_consumption',
    'Miscellaneous Fertilizer Application' : 'crop_acreage'
}

print('Proxies by End-Use Category:')
for market, proxy in market_to_proxy_dict.items():
    print(f' - {market}: {proxy}')

Proxies by End-Use Category:
 - Cement Manufacturing: cement_production
 - Lime Manufacturing: lime_facilities
 - Glass Manufacturing: glass_emissions
 - Gypsum-Based Product Manufacturing: gypsum_facilities
 - Steel Pickling: steel_pickling_facilities
 - Pulp and Paper Manufacturing: pulp_and_paper_facilities
 - Lithium Battery Manufacturing: lithium_battery_capacity
 - Cosmetics and Personal Care Product Manufacturing: cosmetics_employment
 - Organic Chemical Manufacturing: org_chemical_employment
 - Pharmaceutical Manufacturing: pharma_employment
 - Phosphoric Acid Manufacturing: phosphoric_acid_facilities
 - Miscellaneous Chemical Manufacturing: chemical_employment
 - Miscellaneous Manufacturing: manufacturing_employment
 - Industrial Water Treatment: industrial_water_use
 - Drinking Water Treatment: public_supply_water_use
 - Wastewater Treatment: wwtp_flow
 - Deicing and Dust Control: road_salts_application
 - Miscellaneous Construction: construction_spending
 - Oil and Gas Drill

# Helper Functions and Variables

In [47]:
# dictionary mapping each byproduct to its demand data specificity and resolution
calc_method_dict = dict.fromkeys(market_df['byproduct'].unique(), 'Byproduct x Domestic')
calc_method_dict.update(dict.fromkeys(bpms_df['byproduct'].unique(), 'Byproduct-Market x State'))
calc_method_dict

{'BaSO4': 'Byproduct x Domestic',
 'Br': 'Byproduct x Domestic',
 'Ca(OH)2': 'Byproduct x Domestic',
 'CaCO3': 'Byproduct-Market x State',
 'CaSO4': 'Byproduct x Domestic',
 'HCl': 'Byproduct x Domestic',
 'K2O': 'Byproduct x Domestic',
 'Li': 'Byproduct x Domestic',
 'Mg(OH)2': 'Byproduct x Domestic',
 'MgCl2': 'Byproduct-Market x State',
 'MgSO4': 'Byproduct x Domestic',
 'Na2CO3': 'Byproduct x Domestic',
 'NaCl': 'Byproduct x Domestic',
 'NaOH': 'Byproduct x Domestic',
 'P2O5': 'Byproduct x Domestic',
 'SrSO4': 'Byproduct x Domestic'}

In [48]:
# demand columns
demand_cols = ['demand_2021_kt', 'demand_2022_kt', 'demand_2023_kt', 'demand_2024_kt', 'demand_2025_kt']

# price cols
price_cols = ['price_2021_usd_t', 'price_2022_usd_t', 'price_2023_usd_t', 'price_2024_usd_t', 'price_2025_usd_t']

In [49]:
# dictionary mapping state names to state alphas
state_name_to_alpha_dict = dict(zip(st_df['NAME'], st_df['STUSPS']))

In [50]:
def calculate_state_demand(bp, market):
    """
    Calculates the demand for a given byproduct in a given market for all
    states in CONUS using the methods specified in methods_dict. Returns a Series with
    the results.

    Arguments:
    bp : str
        The byproduct for which to calculate demand.
    market : str
        The market for which to calculate demand.
    
    Returns:
    state_demand : pd.DataFrame
        A DataFrame with the estimated state-level demand for the byproduct in the specified marke 
        for each year 2021-2025.
    
    """
    # create a dataframe to hold state-level demand results
    state_demand = pd.DataFrame(index=st_df['STUSPS'], columns=demand_cols)

    # identify calculation method based on byproduct
    calc_method = calc_method_dict[bp]

    # Method #1: Byproduct x Domestic
    if calc_method == 'Byproduct x Domestic':
        # get demand data for this byproduct from bpd_df
        bp_demand = bpd_df.loc[bpd_df['byproduct']==bp][demand_cols].iloc[0].copy()

        # get market share from market_df
        market_share = market_df.loc[(market_df['byproduct']==bp) & (market_df['end_use_category']==market), 'pct_demand'].iloc[0].copy()

        # byproduct-market demand = domestic demand * market share
        bpm_demand = bp_demand * market_share

        # disaggregate to state-level using proxy
        try:
            proxy = market_to_proxy_dict[market]
        except KeyError:
            raise ValueError(f"No proxy found for market: {market}")

        proxy_df = proxy_dict[proxy]
        proxy_df = proxy_df.set_index('State Alpha')
        state_shares = proxy_df.loc[state_demand.index, 'Percent of Total']
        state_demand[demand_cols] = state_shares.values[:, np.newaxis] * bpm_demand[demand_cols].values

    # Method #2: Byproduct-Market x State
    elif calc_method == 'Byproduct-Market x State':
        # get state-level demand data for this byproduc-market from bpms_df
        bpms_demand = bpms_df.loc[(bpms_df['byproduct']==bp) & (bpms_df['end_use_category']==market)].set_index('state', drop=True).copy()

        # get total withheld quantities for each year
        totals_withheld = bpms_demand.loc['Total Withheld', demand_cols].copy()

        # drop rows for states outside of conus
        bpms_demand = bpms_demand[bpms_demand.index.isin(st_df['NAME'])].copy()
        bpms_demand.index = bpms_demand.index.map(state_name_to_alpha_dict)

        # iterate through each year and fill in state_demand
        for col in demand_cols:
            # use state-level demand data directly
            state_demand[col] = bpms_demand.loc[state_demand.index, col]

            # get the total withheld quantity
            total_withheld = totals_withheld[col]
            
            # see if any quantity has been withheld
            if total_withheld > 0:
                
                # get the states with missing demand data
                missing_states = bpms_demand[bpms_demand[col]=='W'].index.tolist()
            
                # get the proxy for this market
                proxy = market_to_proxy_dict[market]
                proxy_df = proxy_dict[proxy]
                proxy_df = proxy_df.set_index('State Alpha')

                # isolate the proxy data for the missing states
                w_proxy_df = proxy_df.loc[missing_states].copy()
                w_proxy_df['Percent of Total'] = w_proxy_df['Metric Value'] / w_proxy_df['Metric Value'].sum() # recalculate percent of total for states with withheld data

                for state in missing_states:
                    # calculate the state-level demand for the withheld quantity using the proxy
                    state_demand.loc[state, col] = w_proxy_df.loc[state, 'Percent of Total'] * total_withheld
            
    else:
        raise ValueError(f"Unknown calculation method: {calc_method}")
    
    # cast all demand values to float
    state_demand = state_demand.astype(float)
    
    return state_demand
            

In [51]:
# dictionary to store results
demand_dict = {}

# Calculate Demand

In [52]:
# iterate through byproducts and markets to calculate state-level demand
for bp in market_df['byproduct'].unique():
    print(f'========== {bp} ==========')
    bp_markets = market_df.loc[market_df['byproduct']==bp, 'end_use_category'].values
    demand_dict[bp] = {}
    print(f'\t End Use Categories:')
    for market in bp_markets:
        print(f'\t\t - {market} ({100*market_df[(market_df["byproduct"]==bp) & (market_df["end_use_category"]==market)]["pct_demand"].values[0]:.1f}%)')
        if market == 'Other':
            continue
        state_demand = calculate_state_demand(bp, market)
        demand_dict[bp][market] = state_demand


========== BaSO4 ==========
	 End Use Categories:
		 - Miscellaneous Manufacturing (10.0%)
		 - Oil and Gas Drilling (90.0%)
========== Br ==========
	 End Use Categories:
		 - Industrial Water Treatment (8.4%)
		 - Miscellaneous Chemical Manufacturing (58.9%)
		 - Oil and Gas Drilling (12.6%)
		 - Organic Chemical Manufacturing (10.5%)
		 - Pharmaceutical Manufacturing (9.5%)
========== Ca(OH)2 ==========
	 End Use Categories:
		 - Drinking Water Treatment (5.6%)
		 - Flue Gas Desulfurization (18.1%)
		 - Miscellaneous Chemical Manufacturing (24.9%)
		 - Miscellaneous Construction (35.6%)
		 - Other (6.2%)
		 - Wastewater Treatment (9.6%)
========== CaCO3 ==========
	 End Use Categories:
		 - Cement Manufacturing (15.7%)
		 - Lime Manufacturing (4.6%)
		 - Other (79.7%)
========== CaSO4 ==========
	 End Use Categories:
		 - Cement Manufacturing (11.9%)
		 - Gypsum-Based Product Manufacturing (49.3%)
		 - Miscellaneous Fertilizer Application (38.8%)
========== HCl ==========
	 End Use 

# Export Results

In [53]:
# compile results into a single dataframe
results_df = pd.DataFrame(columns=['byproduct', 'end_use_category', 'state_alpha', 'year', 'demand_kt', 'price_usd_t'])

for bp, market_demand_dict in demand_dict.items():
    for market, state_demand_df in market_demand_dict.items():
        for year in range(2021, 2026):
            demand_col = f'demand_{year}_kt'
            price_col = f'price_{year}_usd_t'
            temp_df = pd.DataFrame({
                'byproduct': bp,
                'end_use_category': market,
                'state_alpha': state_demand_df.index,
                'year': year,
                'demand_kt': state_demand_df[demand_col].values,
                'price_usd_t': price_df.loc[price_df['byproduct']==bp, price_col].values[0]
            })
            results_df = pd.concat([results_df, temp_df], ignore_index=True)

# export to csv
results_df.to_csv('../data/outputs/results.csv', index=False)

/tmp/ipykernel_2841460/748960015.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, temp_df], ignore_index=True)
